# Space LLM - Training on Google Colab

Custom 10M parameter decoder-only transformer trained on space/astronomy data.

**Features:**
- RoPE, SwiGLU, causal attention, KV-cache
- Data from Wikipedia + arXiv + NASA
- Self-evolution engine (mutates & improves itself)

**Steps:** Run each cell in order. Enable GPU first: Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 1: Clone repo and install dependencies
import os

if not os.path.exists("space-llm"):
    !git clone https://github.com/korfalor-cloud/space-llm.git

%cd space-llm
!pip install -r requirements.txt

# Add to Python path so imports work
import sys
sys.path.insert(0, ".")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Step 2: Prepare space data (Wikipedia + arXiv + NASA)
# Downloads, cleans, tokenizes and saves as binary files
# Takes ~15-30 minutes depending on internet speed
!python data/prepare_data.py

In [ ]:
# Step 3: Train the model
# ~4-8 hours on T4 GPU, checkpoints saved every 5000 steps
# Training will print loss, perplexity, and tokens/sec as it runs
!python train.py

In [ ]:
# Step 4: Generate text (example prompts)
import sys, os
sys.path.insert(0, ".")

from generate import load_model, load_tokenizer, generate_text
from config import GenerateConfig

model, _ = load_model("checkpoints")
tokenizer = load_tokenizer("data/tokenizer/space_tokenizer.model")
gen_config = GenerateConfig(temperature=0.7, top_k=40, top_p=0.85, max_new_tokens=200)

prompts = [
    "Question: What is a black hole?\nAnswer:",
    "Question: How far is Mars from Earth?\nAnswer:",
    "Question: What is dark matter?\nAnswer:",
    "Question: Tell me about the James Webb Space Telescope.\nAnswer:",
    "The Sun is a star that",
]

for prompt in prompts:
    print(f"\n{'='*50}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*50}")
    response = generate_text(model, tokenizer, prompt, gen_config)
    print(response)

In [ ]:
# Step 5: Evaluate - perplexity + sample generations
!python evaluate.py

In [ ]:
# Step 6: Self-Evolution (optional but cool!)
# The model mutates its own architecture, retrains, and keeps the best version
# Runs for N generations, each with M mutations
# Takes several hours - run overnight or on a long Colab session
!python self_evolve.py --generations 5 --mutations-per-gen 3 --train-steps 5000

In [ ]:
# Step 7: Ask your own questions!
import sys
sys.path.insert(0, ".")

from generate import load_model, load_tokenizer, generate_text
from config import GenerateConfig

model, _ = load_model("checkpoints")
tokenizer = load_tokenizer("data/tokenizer/space_tokenizer.model")
gen_config = GenerateConfig(temperature=0.7, top_k=40, top_p=0.85, max_new_tokens=256)

# Change this to any space question you want!
your_question = "What would happen if the Sun suddenly disappeared?"

prompt = f"Question: {your_question}\nAnswer:"
print(f"Q: {your_question}\n")
response = generate_text(model, tokenizer, prompt, gen_config)
print(response)